# Tech Challenge — Olist

## Pergunta norteadora

**O desempenho logístico impacta diretamente a satisfação dos clientes e o potencial de retenção e crescimento das operações de comércio eletrônico?**

Este projeto utiliza o Brazilian E-commerce Dataset disponibilizado pela Olist para analisar a relação entre eficiência logística, satisfação dos consumidores e indicadores de crescimento.

Ao longo do estudo são avaliados aspectos como tempo de aprovação dos pedidos, prazo de entrega, atrasos logísticos, avaliações dos clientes, retenção e evolução da receita, buscando identificar evidências de como a qualidade da operação logística pode influenciar a experiência do consumidor e a sustentabilidade do crescimento do negócio.

In [ ]:
#Importando bibliotecas para análise

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Leitura dos arquivos

customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
translation = pd.read_csv('product_category_name_translation.csv')

## **Análises iniciais**

### Análise inicial de dimensão dos datasets

Realizamos uma análise preliminar das dimensões das tabelas após a leitura dos dataframes. Utilizamos o método df.shape para obter o número de linhas e colunas de cada tabela. Como resultado, identificamos que a tabela de geolocalização possui mais de 1 milhão de registros. Esse dimensionamento será utilizado como base para a próxima etapa, onde iremos focar na limpeza e manipulação dessa tabela, a fim de garantir consistência e integridade dos dados para análises futuras.

In [ ]:
datasets = {
    'customers': customers,
    'orders': orders,
    'items': items,
    'products': products,
    'reviews': reviews,
    'payments': payments,
    'sellers': sellers,
    'geolocation': geolocation,
    'translation': translation,
}

for nome, df in datasets.items():
    print(f'{nome:12s} -> {df.shape[0]:>9,} linhas | {df.shape[1]} colunas')


### Verificação inicial de dados dos datasets

Realizamos uma leitura rápida (head) das principais tabelas (orders, customers, items, products) para entender a estrutura inicial dos dados, verificar o preenchimento das colunas e obter uma visão preliminar de como as informações estão organizadas.

Talvez utilizar df.info() para uma melhor visualização

In [ ]:
# Visão geral rápida das principais tabelas

print('ORDERS')
display(orders.head())

print('CUSTOMERS')
display(customers.head())

print('ITEMS')
display(items.head())

print('PRODUCTS')
display(products.head())

### Análise Preliminar das Tabelas
Nesta etapa, realizamos uma leitura rápida das tabelas principais para compreender a estrutura dos dados. Identificamos que algumas tabelas possuem valores nulos, especialmente nas colunas de datas e descrições. Além disso, na tabela de geolocalização, observamos um alto número de duplicatas, muitas vezes associadas ao mesmo CEP, mas com diferentes coordenadas. Esses pontos serão tratados de forma mais detalhada nas próximas etapas, durante o processo de limpeza e manipulação dos dados.

In [ ]:
# Identificando valores nulos e duplicados nas bases

for nome, df in [('orders', orders), ('geolocation', geolocation), ('customers', customers), ('items', items), ('products', products), ('reviews', reviews), ('payments', payments), ('sellers', sellers)]:
    print(f'\n--- {nome.upper()} ---')
    print('Duplicados:', df.duplicated().sum())
    print(df.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Validação dos tipos de dados das colunas em cada dataset

print("CUSTOMERS DTYPE:")
print(customers.dtypes, '\n')

print("ORDERS DTYPE:")
print(orders.dtypes, '\n')

print("ITEMS DTYPE:")
print(items.dtypes, '\n')

print("PRODUCTS DTYPE:")
print(products.dtypes, '\n')

print("REVIEWS DTYPE:")
print(reviews.dtypes, '\n')

print("PAYMENTS DTYPE:")
print(payments.dtypes, '\n')

print("SELLERS DTYPE:")
print(sellers.dtypes, '\n')

print("GEOLOCATION DTYPE:")
print(geolocation.dtypes, '\n')

print("TRANSLATION DTYPE:")
print(translation.dtypes, '\n')

### **Tratativa dos dataframes**

### Tratativa e Conversão de Tipos de Dados

Nesta etapa, realizamos uma série de conversões de tipos para garantir a consistência e preparar os dados para as análises. Os passos foram:

- **Conversão de Identificadores**: As colunas de identificadores (como `customer_id`, `product_id`, etc.) foram convertidas para o tipo string, garantindo que IDs sejam tratados como texto.

- **Conversão de Inteiros**: Campos que representam contagens ou medidas (como `product_name_lenght`, `product_description_lenght`, `product_photos_qty`) foram convertidos para inteiros, utilizando o tipo `int`, assegurando a precisão para análises numéricas.

- **Conversão de Datas**: As colunas que representam datas (`order_purchase_timestamp`, `order_approved_at`, etc.) foram convertidas para o tipo datetime, permitindo cálculos temporais e análises de sazonalidade.

- **Tratamento de Valores Nulos**: Identificamos colunas com valores nulos, especialmente em `product_category_name`. Para evitar a perda de dados, optamos por preencher essas categorias nulas com o valor "Desconhecida".

Ao final, estas conversões garantem que nossos dados estejam alinhados, prontos para as próximas etapas de análise, evitando erros e inconsistências em etapas subsequentes.

In [ ]:
# Convertendo tipos de dados de cada coluna e tratando valores nulos e duplicados

#===========#
# CUSTOMERS #
#===========#
customers['customer_id'] = customers['customer_id'].astype(str)
customers['customer_unique_id'] = customers['customer_unique_id'].astype(str)
customers['customer_city'] = customers['customer_city'].astype(str)
customers['customer_state'] = customers['customer_state'].astype(str)

#========#
# ORDERS #
#========#
orders['order_id'] = orders['order_id'].astype(str)
orders['customer_id'] = orders['customer_id'].astype(str)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

#=======#
# ITEMS #
#=======#
items['order_id'] = items['order_id'].astype(str)
items['order_item_id'] = items['order_item_id'].astype(int)
items['product_id'] = items['product_id'].astype(str)
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
items['seller_id'] = items['seller_id'].astype(str)

#==========#
# PRODUCTS #
#==========#
products['product_id'] = products['product_id'].astype(str)
products['product_category_name'] = products['product_category_name'].fillna('Desconhecida').astype(str) # Preenche campos que possuem valores nulos com "Desconhecida"

####################

# Validação de impacto de substituir valores vazios na coluna "product_category_name" por "Desconhecida"

# Cria cópia do DataFrame original
products_original = products.copy()

# Cria cópia com 'Desconhecida'
products_desconhecida = products.copy()
products_desconhecida['product_category_name'] = (products_desconhecida['product_category_name'].fillna('Desconhecida'))

# Exemplo de agregação: total de vendas por categoria
vendas_original = products_original.merge(items, on='product_id', how='inner')
vendas_original = vendas_original.groupby('product_category_name').agg({'order_id': 'count', 'price': 'sum'}).reset_index()

vendas_desconhecida = products_desconhecida.merge(items, on='product_id', how='inner')
vendas_desconhecida = vendas_desconhecida.groupby('product_category_name').agg({'order_id': 'count', 'price': 'sum'}).reset_index()

# Comparação
print("Vendas original (com nulos):")
print(vendas_original)

print("Vendas com 'Desconhecida':")
print(vendas_desconhecida)

# Com as saídas deste trecho de código, não identificamos neste momento impactos negativos ou alterações significativas nas análises com a alteração

####################

products['product_name_lenght'] = products['product_name_lenght'].fillna(0).astype(int) # Utilizamos o fillna(0) para preencher campos nulos com 0
products['product_description_lenght'] = products['product_description_lenght'].fillna(0).astype(int)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0).astype(int)
products['product_weight_g'] = products['product_weight_g'].fillna(0).astype(int)
products['product_length_cm'] = products['product_length_cm'].fillna(0).astype(int)
products['product_height_cm'] = products['product_height_cm'].fillna(0).astype(int)
products['product_width_cm'] = products['product_width_cm'].fillna(0).astype(int)

#=========#
# REVIEWS #
#=========#
reviews['review_id'] = reviews['review_id'].astype(str)
reviews['order_id'] = reviews['order_id'].astype(str)
reviews['review_comment_title'] = reviews['review_comment_title'].astype(str)
reviews['review_comment_message'] = reviews['review_comment_message'].astype(str)
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

#==========#
# PAYMENTS #
#==========#
payments['order_id'] = payments['order_id'].astype(str)
payments['payment_type'] = payments['payment_type'].astype(str)

#=========#
# SELLERS #
#=========#
sellers['seller_id'] = sellers['seller_id'].astype(str)
sellers['seller_city'] = sellers['seller_city'].astype(str)
sellers['seller_state'] = sellers['seller_state'].astype(str)

#=============#
# GEOLOCATION #
#=============#
geolocation['geolocation_city'] = geolocation['geolocation_city'].astype(str)
geolocation['geolocation_state'] = geolocation['geolocation_state'].astype(str)

# Realizamos a agregação dos dados da tabela de geolocalização, agrupando os registros pelo prefixo do CEP.
# A análise inicial identificou uma quantidade excessiva de registros, pois cada variação mínima de latitude e longitude gerava múltiplas linhas para o mesmo CEP.
# Ao agrupar, calculamos a média das coordenadas (latitude e longitude) e mantivemos a cidade e o estado de referência.
# Testamos o impacto agregando os dados com e sem o agrupamento e confirmamos que as métricas, como total de vendas, se mantiveram estáveis.
# Dessa forma, a base resultante é mais leve, facilitando manipulações e análises subsequentes, sem impacto significativo nas conclusões.
geo_agrupada = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': 'first',
        'geolocation_state': 'first'
    })
)
print(geo_agrupada.head())

#=============#
# TRANSLATION #
#=============#
translation['product_category_name'] = translation['product_category_name'].astype(str)
translation['product_category_name_english'] = translation['product_category_name_english'].astype(str)


## Análise de Performance Logística

Após a etapa de tratamento e preparação dos dados, iniciamos a análise das métricas logísticas do marketplace.

O objetivo desta etapa é compreender o comportamento operacional dos pedidos ao longo do tempo, identificando padrões de aprovação, entrega e atrasos. Essas métricas são fundamentais para responder à pergunta norteadora do projeto, uma vez que a eficiência logística impacta diretamente a experiência do cliente e, consequentemente, sua satisfação.

Inicialmente serão analisadas três métricas principais:

- Tempo de aprovação do pedido;
- Tempo de entrega ao cliente;
- Dias de atraso em relação à data estimada.

Além da análise estatística, serão investigados possíveis valores extremos (outliers), buscando identificar se representam comportamentos reais do negócio ou inconsistências operacionais capazes de distorcer interpretações futuras.

In [ ]:
# Tempo entre compra e aprovação

orders['tempo_aprovacao_horas'] = (
    orders['order_approved_at'] -
    orders['order_purchase_timestamp']
).dt.total_seconds() / 3600

# Tempo entre compra e entrega

orders['tempo_entrega_dias'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# Diferença entre entrega real e estimada

orders['dias_atraso'] = (
    orders['order_delivered_customer_date'] -
    orders['order_estimated_delivery_date']
).dt.days

### Construção das Métricas Logísticas

Para avaliar a eficiência operacional dos pedidos, foram construídas métricas capazes de representar diferentes etapas da jornada logística.

O tempo de aprovação permite avaliar a agilidade inicial do processamento dos pedidos. O tempo de entrega representa o período total necessário para que o produto chegue ao consumidor. Já a métrica de dias de atraso permite medir o cumprimento do prazo prometido ao cliente.

Esses indicadores serão utilizados ao longo das análises para investigar possíveis impactos da operação logística na satisfação dos consumidores e nos resultados do negócio.

Análise do tempo de aprovação

In [ ]:
orders['tempo_aprovacao_horas'].describe()

In [ ]:
orders[
    ['order_id', 'tempo_aprovacao_horas', 'order_status']
].sort_values('tempo_aprovacao_horas', ascending=False).head(30)

A análise do tempo de aprovação revelou que a maior parte dos pedidos é aprovada em poucas horas após a compra. Entretanto, foram identificados registros extremamente discrepantes, com tempos superiores a 200 horas e casos isolados acima de 4.500 horas.

Considerando que esses registros representam uma parcela extremamente reduzida da base e possuem potencial para distorcer médias e visualizações, optou-se por classificá-los como outliers operacionais e removê-los das análises subsequentes.

Essa decisão permite que os indicadores reflitam com maior precisão o comportamento predominante da operação.

Tempo de entrega

In [ ]:
orders['tempo_entrega_dias'].describe()

In [ ]:
orders[
    ['order_id', 'tempo_entrega_dias', 'order_status']
].sort_values('tempo_entrega_dias', ascending=False).head(30)

O tempo de entrega apresentou comportamento relativamente estável para a maior parte dos pedidos, concentrando-se próximo à mediana observada.

Entretanto, foram identificadas entregas superiores a 100 dias, incluindo casos acima de 200 dias. Embora representem ocorrências reais, esses registros constituem eventos excepcionais dentro da operação e possuem potencial para distorcer análises agregadas.

Dessa forma, optou-se por removê-los da análise principal, preservando o foco no comportamento predominante da experiência de entrega.

In [ ]:
orders['dias_atraso'].describe()

In [ ]:
orders[
    ['order_id', 'dias_atraso', 'order_status']
].sort_values('dias_atraso', ascending=False).head(30)

A análise dos dias de atraso demonstrou que a maioria dos pedidos foi entregue dentro do prazo previsto ou antecipadamente.

Apesar disso, foram identificados casos extremos com atrasos superiores a 100 dias. Por representarem ocorrências excepcionais e pouco representativas da operação como um todo, esses registros foram classificados como outliers.

A remoção desses casos permite uma avaliação mais consistente da capacidade logística da plataforma e de sua relação com a satisfação dos consumidores.

### Remoção de Outliers nas Métricas Logísticas

Após a análise das métricas de tempo de aprovação, tempo de entrega e dias de atraso, definimos que os outliers identificados não representam a realidade da maioria dos pedidos. A seguir, aplicamos a remoção dos registros extremos, garantindo que a análise subsequente seja focada na realidade predominante.

In [ ]:
# Remoção de outliers no tempo de aprovação (acima de 200 horas)
orders = orders[orders['tempo_aprovacao_horas'] <= 200]

# Remoção de outliers no tempo de entrega (acima de 100 dias)
orders = orders[orders['tempo_entrega_dias'] <= 100]

# Remoção de outliers nos dias de atraso (acima de 100 dias)
orders = orders[orders['dias_atraso'] <= 100]

A remoção dos registros extremos resultou em uma base mais representativa do comportamento operacional predominante.

Embora esses casos possam possuir relevância para análises específicas de exceções logísticas, eles não representam o padrão observado na maior parte das transações.

Com a base ajustada, as análises seguintes passam a refletir com maior precisão a experiência típica dos consumidores, permitindo investigar de forma mais confiável a relação entre logística, satisfação e crescimento do negócio.

## KPIs Gerais do Negócio

Antes de aprofundar as análises específicas, é importante compreender a dimensão da operação analisada.

Os indicadores apresentados nesta seção fornecem uma visão consolidada do marketplace, incluindo volume de vendas, receita gerada, alcance geográfico, diversidade de produtos e satisfação dos consumidores.

Esses resultados servirão como referência para contextualizar as análises posteriores e compreender como os indicadores logísticos se relacionam com os resultados do negócio.

Receita geral com frete - tabela 'payments'

In [ ]:
receita_total = (payments['payment_value']).sum()

print(f'Receita Total: R$ {receita_total:,.2f}')

In [ ]:
total_pedidos = orders['order_id'].nunique()

ticket_medio = receita_total / total_pedidos

print(f"Receita Total: R$ {receita_total:,.2f}")
print(f"Ticket Médio: R$ {ticket_medio:,.2f}")

Clientes

In [ ]:
clientes_unicos = customers['customer_unique_id'].nunique()

print(clientes_unicos)

Total de pedidos


In [ ]:
total_pedidos = orders['order_id'].nunique()

print(total_pedidos)

Pedidos entregues

In [ ]:
pedidos_entregues = (
    orders['order_status']
    .eq('delivered')
    .sum()
)

print(pedidos_entregues)

Vendedores ativos

In [ ]:
vendedores_ativos = sellers['seller_id'].nunique()

print(vendedores_ativos)

Média de avaliações

In [ ]:
nota_media = reviews['review_score'].mean()

print(f'Nota média: {nota_media:.2f}')

Tempo médio de entrega

In [ ]:
tempo_medio_entrega = orders['tempo_entrega_dias'].mean()

print(tempo_medio_entrega)

Compliance

In [ ]:
pedidos_entregues = orders[
    orders['order_status'] == 'delivered'
].copy()

compliance = (
    (pedidos_entregues['dias_atraso'] <= 0)
    .mean()
    * 100
)

entregues_no_prazo = (
    pedidos_entregues['dias_atraso'] <= 0
).sum()

entregues_atrasados = (
    pedidos_entregues['dias_atraso'] > 0
).sum()

print(f'Entregues no prazo: {entregues_no_prazo:,}')
print(f'Entregues atrasados: {entregues_atrasados:,}')
print(f'Compliance: {compliance:.2f}%')

Taxa de recompra

In [ ]:
# Merge
df = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

# Frequência de compras
frequencia_clientes = (
    df.groupby('customer_unique_id')['order_id']
      .count()
)

clientes_1_compra = (frequencia_clientes == 1).sum()
clientes_recompra = (frequencia_clientes >= 2).sum()

# Dados
dados = [clientes_1_compra, clientes_recompra]

labels = [
    f'Uma compra ({clientes_1_compra:,})',
    f'Recorrente (2+ compras) ({clientes_recompra:,})'
]


cores = ['#ED145B', '#4A4A4A']

# Total
total = sum(dados)

fig, ax = plt.subplots(figsize=(8, 6))

# Gráfico de rosca
wedges, texts, autotexts = ax.pie(
    dados,
    colors=cores,
    startangle=90,
    autopct='%1.1f%%',
    pctdistance=0.88,
    wedgeprops=dict(
        width=0.45,
        edgecolor='white',
        linewidth=2
    )
)

# Estilo dos percentuais
menor_fatia = dados.index(min(dados))

for i, autotext in enumerate(autotexts):
    autotext.set_color('white')
    autotext.set_fontweight('bold')

    if i == menor_fatia:
        autotext.set_rotation(90)
        autotext.set_fontsize(10)
        autotext.set_va('center')
        autotext.set_ha('center')
    else:
        autotext.set_fontsize(11)

# Texto central
ax.text(
    0,
    0,
    f'{total:,}\nclientes',
    ha='center',
    va='center',
    fontsize=13,
    fontweight='bold'
)

# Título
plt.title(
    'Clientes Únicos vs. Recorrentes',
    fontsize=15,
    fontweight='bold',
    pad=20
)

# Legenda
plt.legend(
    wedges,
    labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.show()

Categorias

In [ ]:
categorias_ativas = (
    products['product_category_name']
    .nunique()
)

print(categorias_ativas)

Ticket médio

In [ ]:
receita_total = items['price'].sum()

total_pedidos = orders['order_id'].nunique()

ticket_medio = receita_total / total_pedidos

print(f'Ticket Médio: R$ {ticket_medio:.2f}')

## Gráficos

Relação de receita total/pedidos por mês

In [ ]:
receita_pedidos_mes = (
    items
    .merge(
        orders[['order_id', 'order_purchase_timestamp']],
        on='order_id',
        how='inner'
    )
)

receita_pedidos_mes['mes_ano'] = (
    receita_pedidos_mes['order_purchase_timestamp']
    .dt.to_period('M')
    .astype(str)
)

receita_pedidos_mes = (
    receita_pedidos_mes
    .groupby('mes_ano')
    .agg(
        receita_total=('price', 'sum'),
        quantidade_pedidos=('order_id', 'nunique')
    )
    .reset_index()
)

display(receita_pedidos_mes.head(20))

In [ ]:
# Formata o eixo X para MM/AAAA
receita_pedidos_mes['mes_ano'] = pd.to_datetime(
    receita_pedidos_mes['mes_ano']
).dt.strftime('%m/%Y')

fig, ax1 = plt.subplots(figsize=(14,6))

# Receita
linha_receita = ax1.plot(
    receita_pedidos_mes['mes_ano'],
    receita_pedidos_mes['receita_total'],
    color='#ED145B',
    marker='o',
    linewidth=2.5,
    label='Receita (R$)'
)

ax1.set_ylabel(
    'Receita (R$)',
    color='#ED145B',
    fontweight='bold'
)

ax1.tick_params(
    axis='y',
    labelcolor='#ED145B'
)

# Grade horizontal e vertical
ax1.grid(
    True,
    axis='both',
    linestyle='--',
    linewidth=0.7,
    alpha=0.5
)

# Quantidade de pedidos
ax2 = ax1.twinx()

linha_pedidos = ax2.plot(
    receita_pedidos_mes['mes_ano'],
    receita_pedidos_mes['quantidade_pedidos'],
    color='#4A4A4A',
    marker='s',
    linestyle='--',
    linewidth=2,
    label='Quantidade de Pedidos'
)

ax2.set_ylabel(
    'Quantidade de Pedidos',
    color='#4A4A4A',
    fontweight='bold'
)

ax2.tick_params(
    axis='y',
    labelcolor='#4A4A4A'
)

# Título
plt.title(
    'Receita e Quantidade de Pedidos por Mês',
    fontsize=15,
    fontweight='bold',
    pad=15
)

# Datas na diagonal
ax1.tick_params(
    axis='x',
    rotation=45
)

# Legenda
linhas = linha_receita + linha_pedidos
labels = [linha.get_label() for linha in linhas]

ax1.legend(
    linhas,
    labels,
    loc='upper left',
    frameon=True
)

plt.tight_layout()
plt.show()

Partindo da nossa primeira hipótese, iremos identificar a quantidade de clientes únicos e recorrentes, a fim de elucidar melhor a taxa de recompra.

In [ ]:
# Merge
df = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

# Frequência de compras
frequencia_clientes = (
    df.groupby('customer_unique_id')['order_id']
      .count()
)

clientes_1_compra = (frequencia_clientes == 1).sum()
clientes_recompra = (frequencia_clientes >= 2).sum()

# Dados
dados = [clientes_1_compra, clientes_recompra]

labels = [
    f'Uma compra ({clientes_1_compra:,})',
    f'Recorrente (2+ compras) ({clientes_recompra:,})'
]


cores = ['#ED145B', '#4A4A4A']

# Total
total = sum(dados)

fig, ax = plt.subplots(figsize=(8, 6))

# Gráfico de rosca
wedges, texts, autotexts = ax.pie(
    dados,
    colors=cores,
    startangle=90,
    autopct='%1.1f%%',
    pctdistance=0.88,
    wedgeprops=dict(
        width=0.45,
        edgecolor='white',
        linewidth=2
    )
)

# Estilo dos percentuais
menor_fatia = dados.index(min(dados))

for i, autotext in enumerate(autotexts):
    autotext.set_color('white')
    autotext.set_fontweight('bold')

    if i == menor_fatia:
        autotext.set_rotation(90)
        autotext.set_fontsize(10)
        autotext.set_va('center')
        autotext.set_ha('center')
    else:
        autotext.set_fontsize(11)

# Texto central
ax.text(
    0,
    0,
    f'{total:,}\nclientes',
    ha='center',
    va='center',
    fontsize=13,
    fontweight='bold'
)

# Título
plt.title(
    'Clientes Únicos vs. Recorrentes',
    fontsize=15,
    fontweight='bold',
    pad=20
)

# Legenda
plt.legend(
    wedges,
    labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.show()

### Relação entre Tempo de Entrega e Satisfação dos Clientes

Após compreender o comportamento logístico da operação, o próximo passo consiste em investigar se o prazo necessário para concluir uma entrega influencia a percepção dos consumidores.

Para isso, os pedidos foram agrupados em faixas de tempo de entrega e comparados com suas respectivas avaliações médias.

In [ ]:
kpi_entrega_satisfacao = (
    orders[['order_id', 'tempo_entrega_dias']]
    .merge(
        reviews[['order_id', 'review_score']],
        on='order_id',
        how='inner'
    )
)

# Cria faixas de entrega

kpi_entrega_satisfacao['faixa_entrega'] = pd.cut(
    kpi_entrega_satisfacao['tempo_entrega_dias'],
    bins=[0, 5, 10, 15, 20, 999],
    labels=[
        'Até 5 dias',
        '6 a 10 dias',
        '11 a 15 dias',
        '16 a 20 dias',
        'Mais de 20 dias'
    ]
)

# Calcula avaliação média por faixa

kpi_entrega_satisfacao = (
    kpi_entrega_satisfacao
    .groupby('faixa_entrega', observed=False)
    .agg(
        review_medio=('review_score', 'mean'),
        total_pedidos=('order_id', 'count')
    )
    .reset_index()
)

display(kpi_entrega_satisfacao)

In [ ]:
cores = [
    '#4A4A4A',
    '#ED145B',
    '#ED145B',
    '#ED145B',
    '#ED145B'
]

fig, ax = plt.subplots(figsize=(12,6))

sns.barplot(
    data=kpi_entrega_satisfacao,
    x='faixa_entrega',
    y='review_medio',
    hue='faixa_entrega',
    palette=cores,
    legend=False,
    ax=ax
)

for i, valor in enumerate(kpi_entrega_satisfacao['review_medio']):
    ax.text(
        i,
        valor + 0.03,
        f'{valor:.2f}',
        ha='center',
        fontweight='bold'
    )

ax.set_title(
    'Tempo de Entrega x Avaliação Média',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel('Faixa de tempo de entrega')
ax.set_ylabel('Review médio')
ax.set_ylim(0,5)

plt.tight_layout()
plt.show()

### Relação entre Notas e atraso na entrega dos pedidos

Em seguida, para validação do cenário proposto, verificaremos a relação entre o atraso no prazo de entrega e as avaliações

In [ ]:
# Junta pedidos e avaliações

satisfacao_logistica = (
    orders[
        [
            'order_id',
            'dias_atraso'
        ]
    ]
    .merge(
        reviews[
            [
                'order_id',
                'review_score'
            ]
        ],
        on='order_id',
        how='inner'
    )
)

# Criação das faixas de atraso

satisfacao_logistica['cenario'] = pd.cut(
    satisfacao_logistica['dias_atraso'],
    bins=[-999, 0, 5, 15, 999],
    labels=[
        'No prazo',
        'Até 5 dias',
        '6 a 15 dias',
        'Mais de 15 dias'
    ]
)

# KPI de satisfação

kpi_satisfacao = (
    satisfacao_logistica
    .groupby('cenario', observed=False)
    .agg(
        review_medio=('review_score', 'mean'),
        total_pedidos=('order_id', 'count')
    )
    .reset_index()
)

display(kpi_satisfacao)

In [ ]:
cores = ['#ED145B', '#4A4A4A']

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(
    data=kpi_satisfacao,
    x='cenario',
    y='review_medio',
    hue='cenario',
    palette=cores * 2,  # repete as cores
    legend=False,
    ax=ax
)

for i, valor in enumerate(kpi_satisfacao['review_medio']):
    ax.text(
        i,
        valor + 0.05,
        f'{valor:.2f}',
        ha='center',
        fontsize=10,
        fontweight='bold'
    )

ax.set_title(
    'Avaliação Média por Faixa de Atraso',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel('Faixa de atraso')
ax.set_ylabel('Review médio')
ax.set_ylim(0, 5)

plt.tight_layout()
plt.show()

### Receita mensal por região



In [ ]:
from matplotlib.ticker import FuncFormatter

# Receita por estado
receita_estado = (
    orders
    .merge(customers, on='customer_id')
    .merge(items[['order_id', 'price']], on='order_id')
)

receita_estado = (
    receita_estado
    .query("order_status == 'delivered'")
    .groupby('customer_state')['price']
    .sum()
    .reset_index()
)

# Top 5 maiores
top5 = (
    receita_estado
    .sort_values('price', ascending=False)
    .head(5)
)

# Top 5 menores
bottom5 = (
    receita_estado
    .sort_values('price', ascending=True)
    .head(5)
)

# Adiciona categoria
top5['grupo'] = 'Top 5'
bottom5['grupo'] = 'Bottom 5'

# Junta
top_bottom = pd.concat([top5, bottom5])

# Ordena para visualização
top_bottom = top_bottom.sort_values('price')

# Define cores fixas
cores = [
    '#4A4A4A' if grupo == 'Bottom 5'
    else '#ED145B'
    for grupo in top_bottom['grupo']
]

# Gráfico
fig, ax = plt.subplots(figsize=(12,6))

sns.barplot(
    data=top_bottom,
    x='price',
    y='customer_state',
    palette=cores,
    ax=ax
)

# Remove notação científica
ax.ticklabel_format(style='plain', axis='x')

# Formata eixo em milhões
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'R$ {x/1_000_000:.1f}M')
)

# Valores sobre as barras
for i, valor in enumerate(top_bottom['price']):
    ax.text(
        valor + (top_bottom['price'].max() * 0.01),
        i,
        f'R$ {valor:,.0f}'.replace(',', '.'),
        va='center',
        fontsize=9,
        fontweight='bold'
    )

ax.set_title(
    'Receita por Estado (Top 5 Maiores e Top 5 Menores)',
    fontsize=15,
    fontweight='bold'
)

ax.set_xlabel('Receita Total')
ax.set_ylabel('Estado')

ax.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

sns.despine(
    left=False,
    bottom=False
)

plt.tight_layout()
plt.show()

Quantidade de pedidos por região

In [ ]:
# Quantidade de pedidos por estado

pedidos_estado = (
    orders
    .merge(customers, on='customer_id')
    .query("order_status == 'delivered'")
    .groupby('customer_state')['order_id']
    .nunique()
    .reset_index(name='quantidade_pedidos')
)

# Top 5 maiores
top5 = (
    pedidos_estado
    .sort_values('quantidade_pedidos', ascending=False)
    .head(5)
)

# Top 5 menores
bottom5 = (
    pedidos_estado
    .sort_values('quantidade_pedidos', ascending=True)
    .head(5)
)

# Junta os grupos
top_bottom = pd.concat([top5, bottom5])

# Ordena para visualização
top_bottom = top_bottom.sort_values('quantidade_pedidos')

# Cores
cores = [
    '#4A4A4A' if x < top_bottom['quantidade_pedidos'].median()
    else '#ED145B'
    for x in top_bottom['quantidade_pedidos']
]

plt.figure(figsize=(12,6))

ax = sns.barplot(
    data=top_bottom,
    x='quantidade_pedidos',
    y='customer_state',
    palette=cores,
    hue='customer_state',
    legend=False
)

# Valores nas barras
for i, valor in enumerate(top_bottom['quantidade_pedidos']):
    ax.text(
        valor + (top_bottom['quantidade_pedidos'].max() * 0.01),
        i,
        f'{valor:,.0f}'.replace(',', '.'),
        va='center',
        fontsize=10
    )

plt.title(
    'Quantidade de Pedidos por Estado\n(Top 5 Maiores e Top 5 Menores)',
    fontsize=15,
    fontweight='bold',
    pad=20
)

plt.xlabel('Quantidade de Pedidos')
plt.ylabel('Estado')

plt.grid(
    axis='x',
    linestyle='--',
    alpha=0.4
)

# Remove molduras superiores e direitas
sns.despine(top=True, right=True)

plt.tight_layout()
plt.show()

In [ ]:
# Cria coluna de ano
orders['ano'] = orders['order_purchase_timestamp'].dt.year

# Base de receita
receita_pedido = (
    orders
    .merge(items[['order_id', 'price']], on='order_id')
    .query("order_status == 'delivered'")
)

# KPIs anuais
kpi_anual = (
    receita_pedido
    .groupby('ano')
    .agg(
        receita_total=('price', 'sum'),
        quantidade_pedidos=('order_id', 'nunique')
    )
    .reset_index()
)

# Receita em milhões
kpi_anual['receita_milhoes'] = (
    kpi_anual['receita_total'] / 1_000_000
).round(2)

# Formatação para exibição
kpi_anual['receita_total'] = (
    kpi_anual['receita_total']
    .map(lambda x: f'R$ {x:,.2f}')
    .str.replace(',', 'X')
    .str.replace('.', ',')
    .str.replace('X', '.')
)

display(
    kpi_anual[
        [
            'ano',
            'quantidade_pedidos',
            'receita_total',
            'receita_milhoes'
        ]
    ]
)

## Resposta à Pergunta Norteadora

Os resultados obtidos indicam que o desempenho logístico possui relação direta com a satisfação dos consumidores.

Pedidos entregues com maior atraso apresentaram avaliações inferiores, evidenciando que a qualidade da experiência logística influencia a percepção dos clientes.

Além disso, considerando os baixos índices de recompra observados, melhorias logísticas podem contribuir para elevar a retenção e reduzir a dependência da aquisição contínua de novos consumidores.

Dessa forma, conclui-se que a logística representa um fator estratégico para a satisfação dos clientes e para o crescimento sustentável da operação.

## Exportação de datframes tratatos

In [ ]:
# Exporta tabelas novas com alterações
customers.to_csv('customers.csv', index=False)
orders.to_csv('orders.csv', index=False)
items.to_csv('items.csv', index=False)
products.to_csv('products.csv', index=False)
reviews.to_csv('reviews.csv', index=False)
payments.to_csv('payments.csv', index=False)
sellers.to_csv('sellers.csv', index=False)

# Geolocalização tratada
geo_agrupada.to_csv('geolocation.csv', index=False)

translation.to_csv('translation.csv', index=False)

print('Exportação concluída.')